In [ ]:
# ============================================================
# Cell 0: 环境自检（本章纯 CPU 即可，无需 GPU）
# ============================================================
# 本章只做小张量运算 + 训一个 word2vec + 读取模型配置文件，全程 CPU。
# 所以这里不强制 GPU，只打印环境信息确认 PyTorch 可用。
import sys, platform
import torch

print("Python:", sys.version.split()[0])
print("平台:", platform.platform())
print("PyTorch:", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available(), "（本章用不到，CPU 即可）")

In [ ]:
%%capture
# ============================================================
# Cell 1: 安装依赖
# ============================================================
# %%capture 必须是 cell 第一行，把 pip 安装日志藏起来
# torch:         张量运算——Colab 默认已装且够新，故意【不】加 -U：会话中途
#                升级 torch 会让内核半新半旧（新版 torch/_dynamo/config.py 调
#                Config(deprecated=...)，但已加载的旧 torch.utils._config_module
#                .Config 不认这个参数），随后 import transformers 触发它 import
#                torch 时直接 TypeError。本章只用基础张量运算，无需新版 torch。
# matplotlib:    画 sinusoidal / ALiBi 的热力图
# gensim:        训练 word2vec（自带 text8 语料下载器）
# transformers:  仅用于读 Qwen3 的配置文件（AutoConfig，不下权重），>=4.51 支持 Qwen3
!pip install -q -U matplotlib gensim "transformers>=4.51"

In [ ]:
# ============================================================
# Cell 2: token embedding —— 按 id 查表（对应第 1.1 / 1.2 节）
# ============================================================
import torch
import torch.nn as nn

torch.manual_seed(0)                 # 固定随机种子，结果可复现

V, d = 10, 4                         # 玩具尺寸：词表 10 个 token，每个 embedding 4 维
emb = nn.Embedding(V, d)             # 内部持有一个 [V, d] 的可学习矩阵 E
print("embedding 矩阵 E 的形状:", emb.weight.shape)   # [10, 4] = [V, d]

ids = torch.tensor([[2, 5, 5, 1]])   # 一个 batch、长度 4 的序列，shape [B=1, L=4]
out = emb(ids)                       # 查表：每个 id 取 E 的对应行
print("查表输出形状:", out.shape)     # [1, 4, 4] = [B, L, d]

# 关键：ids 里两个 5 取到的是同一行 —— embedding 只看 id、不看位置
print("两个位置上的 '5' 向量是否相同:", torch.equal(out[0, 1], out[0, 2]))  # True
print("→ 这就是为什么还需要位置编码：同一 token 在不同位置拿到的向量一样")

In [ ]:
# ============================================================
# Cell 3: weight tying —— 输出层复用 embedding 权重（对应第 1.3 节）
# ============================================================
# lm_head 把隐藏向量 h ([*, d]) 映射到词表分数 logits ([*, V])，
# 权重形状 [d, V]，恰好是 embedding 矩阵 E ([V, d]) 的转置。
lm_head = nn.Linear(d, V, bias=False)   # 一个 [d, V] 的线性层（不带 bias）

# 绑定：让 lm_head 的权重 = embedding 权重（PyTorch 中 Linear.weight 形状是 [V, d]，
# 内部按 x @ W.T 计算，所以直接令 lm_head.weight = emb.weight 即实现 W_out = E^T）
lm_head.weight = emb.weight
print("emb.weight 形状:", emb.weight.shape)        # [10, 4]
print("lm_head.weight 形状:", lm_head.weight.shape)  # [10, 4]（同一份张量）
print("两者是否同一个张量:", lm_head.weight is emb.weight)  # True

# 验证：拿一个隐藏向量过 lm_head，得到词表上的 logits
h = torch.randn(1, 4, d)              # 假装是 Transformer 输出的隐藏向量 [B, L, d]
logits = lm_head(h)                   # -> [B, L, V]
print("logits 形状:", logits.shape)   # [1, 4, 10] = [B, L, V]
print("→ 绑定后，输入查表和输出打分共用一套向量语义，省下一份参数")

In [ ]:
# ============================================================
# Cell 4: 训练一个 word2vec 词向量（对应第 2.2 / 2.3 节）
# ============================================================
# 语料用 text8（英文维基百科前 1 亿字符，约 17M 词），gensim 自带下载器一键拉取。
# 训练在 CPU 上约几分钟（取决于核数）——这是本章唯一耗时超过秒级的 cell。
import time
import gensim.downloader as api
from gensim.models import Word2Vec
from gensim.models.callbacks import CallbackAny2Vec

print("下载 text8 语料（约 30 MB 压缩包，首次运行需联网）...")
corpus = api.load("text8")          # 返回一个可迭代的句子语料

EPOCHS = 5                           # 训练轮数，抽成变量给下面的进度回调引用

class EpochLogger(CallbackAny2Vec):
    """进度回调：gensim 默认训练时一声不吭，而这一步要跑好几分钟，很容易让人
    以为卡死了。注册这个回调，每个 epoch 开始/结束各打一行，既看得见它在动、
    又能从单轮耗时估出大概还要等多久。
    （注：gensim 多线程下的 loss 统计不准，所以这里只按 epoch + 用时展示进度，不打 loss。）"""
    def __init__(self, total_epochs):
        self.epoch = 0
        self.total = total_epochs
        self.t0 = None               # 记录本轮开始时间，用来算单轮耗时
    def on_epoch_begin(self, model):
        self.t0 = time.time()
        print(f"  epoch {self.epoch + 1}/{self.total} 训练中...", flush=True)
    def on_epoch_end(self, model):
        self.epoch += 1
        dt = time.time() - self.t0
        print(f"  epoch {self.epoch}/{self.total} 完成，用时 {dt:.1f}s", flush=True)

print("开始训练 word2vec（skip-gram + 负采样）...")
t_start = time.time()
model = Word2Vec(
    corpus,
    vector_size=100,     # 每个词向量 100 维
    window=5,            # 上下文窗口半径 5
    min_count=5,         # 出现少于 5 次的词忽略
    sg=1,                # sg=1 用 skip-gram；sg=0 则用 CBOW
    negative=10,         # 负采样个数 k=10
    workers=4,           # 并行线程
    epochs=EPOCHS,
    callbacks=[EpochLogger(EPOCHS)],   # 注册上面的进度回调
)
wv = model.wv
print(f"训练完成，总用时 {time.time() - t_start:.1f}s，词表大小:", len(wv))
print("'king' 的词向量维度:", wv["king"].shape)   # (100,)

In [ ]:
# ============================================================
# Cell 5: 验证语义结构 —— king - man + woman ≈ queen（对应第 2.3 节）
# ============================================================
# most_similar(positive=[...], negative=[...]) 算的就是 king - man + woman，
# 再在词表里找和这个结果向量余弦最接近的词。
print("king - man + woman ≈ ?")
for word, score in wv.most_similar(positive=["king", "woman"], negative=["man"], topn=5):
    print(f"  {word:12s} {score:.3f}")

# 语义相近 → 向量相近
print("\n和 'cat' 最相近的词:")
for word, score in wv.most_similar("cat", topn=5):
    print(f"  {word:12s} {score:.3f}")

# 另一个类比：首都关系 paris - france + italy ≈ rome
print("\nparis - france + italy ≈ ?")
for word, score in wv.most_similar(positive=["paris", "italy"], negative=["france"], topn=3):
    print(f"  {word:12s} {score:.3f}")

In [ ]:
# ============================================================
# Cell 6: 自注意力对顺序「视而不见」（对应第 3.1 节）
# ============================================================
import torch.nn.functional as F

def toy_self_attention(x):
    """最小自注意力：直接拿输入向量两两点积当分数，softmax 后加权求和。
    x: [L, d]  ->  返回 [L, d]。故意不带任何 Q/K/V 投影和位置编码，
    就是为了暴露『分数只由向量点积决定、与顺序无关』这一点。"""
    scores = x @ x.T                  # [L, L]，score[i,j] = x_i · x_j
    weights = F.softmax(scores, dim=-1)
    return weights @ x                # [L, d]

torch.manual_seed(0)
x = torch.randn(3, 4)                 # 3 个 token，每个 4 维，记作 (a, b, c)
out = toy_self_attention(x)

perm = [2, 0, 1]                      # 把顺序打乱成 (c, a, b)
x_perm = x[perm]
out_perm = toy_self_attention(x_perm)

# 对原输出做同样的置换，应当和「打乱后再算」逐元素相等
print("打乱输入后，注意力输出是否只是跟着换位（数值不变）:",
      torch.allclose(out[perm], out_perm, atol=1e-6))   # True
print("→ 不注入位置，attention 把序列当成无序集合：‘狗咬人’和‘人咬狗’没区别")

# 注：这个 demo 是双向、无掩码注意力，置换等变才严格成立。decoder-only 的因果
#     掩码会打破这种对称、自带一点位置信息，但精确距离仍要靠位置编码补（见第 3.1 节）。

In [ ]:
# ============================================================
# Cell 7: sinusoidal 位置编码 + 热力图（对应第四节）
# ============================================================
import math
import matplotlib.pyplot as plt

def sinusoidal_pe(max_len, d, base=10000.0):
    """返回 [max_len, d] 的 sinusoidal 位置编码矩阵。
    偶数维用 sin、奇数维用 cos，频率按 base^{-2i/d} 几何递减。"""
    pe = torch.zeros(max_len, d)
    pos = torch.arange(max_len).unsqueeze(1)               # [max_len, 1]
    # div_term = base^{-2i/d}，i = 0,1,...,d/2-1 -> 每对 (2i, 2i+1) 的频率
    i = torch.arange(0, d, 2)                               # [d/2]
    div_term = torch.exp(-(math.log(base)) * i / d)         # = base^{-2i/d}
    pe[:, 0::2] = torch.sin(pos * div_term)                # 偶数维
    pe[:, 1::2] = torch.cos(pos * div_term)                # 奇数维
    return pe

pe = sinusoidal_pe(max_len=64, d=32)
print("位置编码矩阵形状:", pe.shape)        # [64, 32] = [max_len, d]
print("数值范围:", float(pe.min()), "~", float(pe.max()))  # 落在 [-1, 1]

# 画热力图：横轴 = 维度，纵轴 = 位置；能看到沿维度方向频率渐变的条纹
plt.figure(figsize=(7, 5))
plt.imshow(pe.numpy(), aspect="auto", cmap="RdBu")
plt.xlabel("embedding dimension")        # matplotlib 标签一律用英文，避免豆腐块
plt.ylabel("position")
plt.title("Sinusoidal Positional Encoding")
plt.colorbar(label="value")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 8: learned 位置编码 = token_emb + pos_emb（对应第五节）
# ============================================================
V, d, max_len = 10, 4, 16
token_emb = nn.Embedding(V, d)        # token id -> 向量
pos_emb   = nn.Embedding(max_len, d)  # 位置 id  -> 向量（新增的这张可学习表）

ids = torch.tensor([[2, 5, 5, 1]])    # [B, L]
B, L = ids.shape
positions = torch.arange(L).unsqueeze(0)          # [[0,1,2,3]], shape [1, L]

x = token_emb(ids) + pos_emb(positions)           # 逐元素相加 -> [B, L, d]
print("token_emb 输出:", token_emb(ids).shape)     # [1, 4, 4]
print("pos_emb 输出:  ", pos_emb(positions).shape)  # [1, 4, 4]
print("相加后:       ", x.shape)                    # [1, 4, 4]

# 关键对比：现在两个位置上的 '5' 不再相同了 —— 因为加了不同的位置向量
print("加位置后，两个 '5' 是否还相同:", torch.equal(x[0, 1], x[0, 2]))  # False
print("→ 位置编码补回了 embedding 丢掉的顺序信息")
print("注意：learned 表只有 max_len =", max_len, "行，超过就越界（硬上限）")

In [ ]:
# ============================================================
# Cell 9: RoPE 旋转位置编码 + 相对位置性质验证（对应第六节）
# ============================================================
def rope_angles(seq_len, d, base=10000.0):
    """每个位置、每对维度的旋转角度，返回 cos / sin，形状均 [seq_len, d]。
    第 i 对维度的角速度 theta_i = base^{-2i/d}（和 sinusoidal 同款频率谱）。"""
    i = torch.arange(0, d, 2).float()              # [d/2]
    theta = base ** (-i / d)                       # [d/2] 角速度
    pos = torch.arange(seq_len).float().unsqueeze(1)  # [seq_len, 1]
    angles = pos * theta                           # [seq_len, d/2] = m * theta_i
    # 每对 (2i, 2i+1) 共享同一角度，所以沿最后一维重复一遍 -> [seq_len, d]
    cos = torch.cos(angles).repeat_interleave(2, dim=-1)
    sin = torch.sin(angles).repeat_interleave(2, dim=-1)
    return cos, sin

def rotate_half(x):
    """把相邻成对维度 (x0,x1,x2,x3,...) 变成 (-x1,x0,-x3,x2,...)，
    配合 cos/sin 即可实现每一对的二维旋转，无需显式构造旋转矩阵。"""
    x1 = x[..., 0::2]      # 偶数维 x0, x2, ...
    x2 = x[..., 1::2]      # 奇数维 x1, x3, ...
    return torch.stack((-x2, x1), dim=-1).flatten(-2)

def apply_rope(x, cos, sin):
    """对 x ([seq_len, d]) 施加 RoPE 旋转，返回同形状张量。"""
    return x * cos + rotate_half(x) * sin

torch.manual_seed(0)
d = 8
seq_len = 16
cos, sin = rope_angles(seq_len, d)

q = torch.randn(d)        # 一个 query 向量
k = torch.randn(d)        # 一个 key 向量

def rope_score(m, n):
    """位置 m 的 q、位置 n 的 k，各自旋转后做点积。"""
    qm = apply_rope(q.unsqueeze(0), cos[m:m+1], sin[m:m+1])[0]
    kn = apply_rope(k.unsqueeze(0), cos[n:n+1], sin[n:n+1])[0]
    return float(qm @ kn)

# 相对距离都是 1 的几组 (m, n)，分数应当几乎相等
print("相对距离 = 1 的几组（应当几乎相等）:")
for m, n in [(0, 1), (3, 4), (10, 11)]:
    print(f"  m={m:2d}, n={n:2d}  ->  score = {rope_score(m, n):.6f}")

# 相对距离不同，分数才不同
print("相对距离不同（应当不同）:")
for m, n in [(0, 1), (0, 2), (0, 5)]:
    print(f"  m={m:2d}, n={n:2d}  (距离 {n-m})  ->  score = {rope_score(m, n):.6f}")

In [ ]:
# ============================================================
# Cell 10: ALiBi 线性偏置矩阵 + 可视化（对应第七节）
# ============================================================
def alibi_bias(seq_len, slope):
    """返回 [seq_len, seq_len] 的因果 ALiBi 偏置矩阵。
    bias[i, j] = -slope * (i - j)  （j <= i）；j > i 的未来位置置 -inf 屏蔽。"""
    i = torch.arange(seq_len).unsqueeze(1)     # [L, 1]
    j = torch.arange(seq_len).unsqueeze(0)     # [1, L]
    dist = i - j                               # [L, L]，下三角为距离、上三角为负
    bias = -slope * dist                       # 越远（dist 越大）惩罚越多
    bias = bias.masked_fill(j > i, float("-inf"))  # 因果屏蔽未来 token
    return bias

seq_len = 8
bias = alibi_bias(seq_len, slope=0.5)
print("ALiBi 偏置矩阵 (slope=0.5)，-inf 表示因果屏蔽的未来位置:")
print(bias)

# 可视化：把 -inf 换成 NaN 便于画图（留白），看下三角越往左下惩罚越深
plt.figure(figsize=(5, 4))
plot_bias = bias.clone()
plot_bias[plot_bias == float("-inf")] = float("nan")
plt.imshow(plot_bias.numpy(), cmap="viridis")
plt.xlabel("key position j")
plt.ylabel("query position i")
plt.title("ALiBi Bias (slope=0.5)")
plt.colorbar(label="bias added to score")
plt.tight_layout()
plt.show()
print("→ 同一行里，j 离 i 越远（越往左），加的惩罚越负，注意力权重越小")

In [ ]:
# ============================================================
# Cell 11: 看真实模型 Qwen3 的位置编码配置（对应第 6.3 / 八节）
# ============================================================
# AutoConfig 只读 config.json（几 KB），不下载十几 GB 权重，CPU 秒级完成
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained("Qwen/Qwen3-8B")
print("模型类型:", cfg.model_type)
print("hidden_size (d_model):", cfg.hidden_size)
print("注意力头数:", cfg.num_attention_heads)
# 每头维度（也就是 6.3 节那个被 RoPE 旋转的 d）优先读 config 里的 head_dim 字段；
# 很多模型确实满足 head_dim == hidden_size / num_heads，但这不是铁律——有些模型
# （含部分 Qwen3 配置）会把 head_dim 单独设成别的值，所以别想当然用除法去推。
head_dim = getattr(cfg, "head_dim", None) or cfg.hidden_size // cfg.num_attention_heads
print("每个头的维度 head_dim:", head_dim)
print("最大位置长度 max_position_embeddings:", cfg.max_position_embeddings)
# RoPE 的 base：transformers 5.x 起把它收进 cfg.rope_parameters 这个 dict（键 rope_theta /
# rope_type），旧版本则是顶层的 cfg.rope_theta；下面两种都兜底，免得换个版本就 AttributeError
rope_params = getattr(cfg, "rope_parameters", None) or {}
rope_theta = rope_params.get("rope_theta") or getattr(cfg, "rope_theta", None)
print("RoPE 的 base (rope_theta):", rope_theta)   # 就是第 6.3 节那个调上下文的旋钮
print("\n→ Qwen3 用 RoPE：没有单独的 learned 位置表，位置信息靠旋转 q/k 注入；")
print("  想扩上下文时，rope_theta 是关键旋钮之一（配合 YaRN 等外推技巧）。")